# Dialogue Summarize

### Install necessary libraries

Install the required packages to use PyTorch and Hugging Face transformers and datasets.

In [1]:
# Check Amazon SageMaker kernel

import sys
print(sys.version)

3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]


In [2]:
!pip install --upgrade pip
!pip install --disable-pip-version-check \
    torch==1.13.1 \
    torchdata==0.5.1 --quiet

!pip install \
    transformers==4.27.2 \
    datasets==2.11.0 --quiet

!pip install pandas==2.1.4 \
    matplotlib==3.8.4 \
    numpy ==1.26.4 \
    rouge_score==0.1.2 --quiet

  Using cached pip-25.2-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 23.3.2
    Uninstalling pip-23.3.2:
      Successfully uninstalled pip-23.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 0.8.3 requires pandas<1.6,>=1.4.1, but you have pandas 2.1.4 which is incompatible.
autogluon-multimodal 0.8.3 requires pytorch-lightning<1.10.0,>=1.9.0, but you have pytorch-lightning 2.0.9 which is incompatible.
autogluon-multimodal 0.8.3 requires scikit-learn<1.4.1,>=1.1, but you have scikit-learn 1.4.2 which is incompatible.
autogluon-multimodal 0.8.3 requires torchmetrics<0.12.0,>=0.11.0, but you have torchmetrics 1.0.3 which is incompatible.
autogluon-multimodal 0.8.3 requires torchvision<0.15.0, but you have torchvision 0.15.2a0+ab7b3e6 whic

In [3]:
import evaluate
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

2025-08-07 12:05:36.023152: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Load components

Load the DialogSum dataset, the pre-trained Large Language Model (LLM) FLAN-T5 from Hugging Face, tokenizer, and configurator.

The dataset contains 10000+ dialogues with the corresponding manually labeled summaries and topics.

In [4]:
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

Found cached dataset csv (/home/studio-lab-user/.cache/huggingface/datasets/knkarthick___csv/knkarthick--dialogsum-cd36827d3490488d/0.0.0/6954658bab30a358235fa864b05cf819af0e179325c740e4bc853bcc7ec513e1)


  0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
print(f"Dataset structure: {dataset}")
print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"Test examples: {len(dataset['test'])}")

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
})
Training examples: 12460
Validation examples: 500
Test examples: 1500


In [6]:
example_indices =  [40, 200]
dash_line = "-".join('' for x in range(100))

for i, index in enumerate(example_indices):
    print(dash_line)
    print("Example", i + 1)
    print(dash_line)
    print("Input Dialogue:")
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print("Baseline Human Summary")
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
Input Dialogue:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
---------------------------------------------------------------------------------------------------
Baseline Human Summary
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------

---------------------------------------------------------------------------------------------------
Examp

Load tge FLAN-T5 model, creating an instance of the AutoModelForSeq2SeqLM class with the .from_pretrained() method.

In [7]:
model_name = "google/flan-t5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/opt/conda/envs/sagemaker-distribution/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/conda/envs/sagemaker-distribution/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [9]:
sentence = "What time is it, Tom?"

sentence_encoded = tokenizer(sentence, return_tensors='pt')

sentence_decoded = tokenizer.decode(
    sentence_encoded['input_ids'][0],
    skip_special_tokens=True
)

print(f"Encoded sentence: {sentence_encoded['input_ids'][0]}\n")
print(f"Decoded sentence: {sentence_decoded}")

Encoded sentence: tensor([ 363,   97,   19,   34,    6, 3059,   58,    1])

Decoded sentence: What time is it, Tom?


In [10]:
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']
    inputs = tokenizer(dialogue, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs['input_ids'],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )
    print(dash_line)
    print("Example", i + 1)
    print(dash_line)
    print("Input Dialogue:")
    print(dialogue)
    print(dash_line)
    print("Baseline Human Summary")
    print(summary)
    print(dash_line)
    print("Model Generated Summary")
    print(output)
    print(dash_line)
    print()
    

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
Input Dialogue:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
---------------------------------------------------------------------------------------------------
Baseline Human Summary
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
Model Generated Summary
Person1: It's ten to nine.
-------------------------------------------------------

In [11]:
# Zero-shot Prompting

def zero_shot_summarization(dialogue):
    prompt = f"Summarize the following dialogue:\n\n{dialogue}\n\nSummary:"
    
    inputs = tokenizer(prompt, return_tensors='pt', max_length=1024, truncation=True)
    output = model.generate(
        inputs.input_ids,
        max_length = 150,
        min_length = 50,
        temperature = 0.7
    )[0]
    summary = tokenizer.decode(output, skip_special_tokens=True)
    
    return summary

# One-shot Prompting

def one_shot_summarization(dialogue):
    example_dialogue = dataset['train'][50]['dialogue']
    example_summary = dataset['train'][50]['summary']
    
    prompt = f"Summarize the following dialogue:{example_dialogue}\n\nSummary\n\n{example_summary}\n\nNow summarize the dialogue\n\n{dialogue}\n\nSummary:"""
    
    inputs = tokenizer(prompt, return_tensors='pt', max_length=1024, truncation=True)
    output = model.generate(
        inputs.input_ids,
        max_length = 150,
        min_length = 50,
        temperature = 0.7
    )[0]
    summary = tokenizer.decode(output, skip_special_tokens=True)
    
    return summary

# Few-shot Prompting

def few_shot_summarization(dialogue):
    example_indices = [20, 50, 80]
    examples = [dataset['train'][i] for i in example_indices]
    
    prompt = "Summarize these dialogues:\n\n"
    
    for ex in examples:
        prompt += f"Dialogue:\n{ex['dialogue']}\n\nSummary:\n{ex['summary']}\n\n"
    
    prompt += f"Dialogue:\n{dialogue}\n\nSummary:"
    
    inputs = tokenizer(prompt, return_tensors='pt', max_length=1024, truncation=True)
    output = model.generate(
        inputs.input_ids,
        max_length = 150,
        min_length = 50,
        temperature = 0.7
    )[0]
    summary = tokenizer.decode(output, skip_special_tokens=True)
    
    return summary

In [12]:
test_indices = [2, 4, 8, 16, 32]
test_examples = [dataset['test'][i] for i in test_indices]

results = []

for i, example in enumerate(test_examples):
    dialogue = example['dialogue']
    summary = example['summary']
    
    print(f"Processing examples {i+1}/{len(test_examples)}...")
    print("Running  zero-shot summarization...")
    zero_shot = zero_shot_summarization(dialogue)
    print("Running  one-shot summarization...")
    one_shot = one_shot_summarization(dialogue)
    print("Running  few-shot summarization...")
    few_shot = few_shot_summarization(dialogue)
    
    results.append({
        "Dialogue": dialogue.strip(),
        "Summary": summary.strip(),
        "Zero-shot Summary": zero_shot.strip(),
        "One-shot Summary": one_shot.strip(),
        "Few-shot Summary": few_shot.strip()
    })
    
    print()

print("Finished processing succesfully!")

results_df = pd.DataFrame(results)

for i, row  in results_df.iterrows():
    print(f"Example {i+1}:")
    print(dash_line)
    print(row["Dialogue"][:150] + "...")
    print("\nSummary:")
    print(dash_line)
    print(row["Summary"])
    print("\nZero-shot Summary:")
    print(dash_line)
    print(row["Zero-shot Summary"])
    print("\nOne-shot Summary:")
    print(dash_line)
    print(row["One-shot Summary"])
    print("\nFew-shot Summary:")
    print(dash_line)
    print(row["Few-shot Summary"])
    print("\n\n")

Processing examples 1/5...
Running  zero-shot summarization...
Running  one-shot summarization...
Running  few-shot summarization...

Processing examples 2/5...
Running  zero-shot summarization...
Running  one-shot summarization...
Running  few-shot summarization...

Processing examples 3/5...
Running  zero-shot summarization...
Running  one-shot summarization...
Running  few-shot summarization...

Processing examples 4/5...
Running  zero-shot summarization...
Running  one-shot summarization...
Running  few-shot summarization...

Processing examples 5/5...
Running  zero-shot summarization...
Running  one-shot summarization...
Running  few-shot summarization...

Finished processing succesfully!
Example 1:
---------------------------------------------------------------------------------------------------
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to al...

Summary:
----------------------

### Evaluate

In [13]:
rouge = evaluate.load('rouge')

def compute_rouge_scores(predictions, references):
    results = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    return results

zero_shot_scores = compute_rouge_scores(
    results_df["Zero-shot Summary"].tolist(), 
    results_df["Summary"].tolist()
)

one_shot_scores = compute_rouge_scores(
    results_df["One-shot Summary"].tolist(), 
    results_df["Summary"].tolist()
)

few_shot_scores = compute_rouge_scores(
    results_df["Few-shot Summary"].tolist(), 
    results_df["Summary"].tolist()
)

metrics = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
rouge_results_df = pd.DataFrame({
    'Metric': metrics,
    'Zero-shot': [zero_shot_scores[m] for m in metrics],
    'One-shot': [one_shot_scores[m] for m in metrics],
    'Few-shot': [few_shot_scores[m] for m in metrics]
})

print("ROUGE Scores Comparison:")
print(rouge_results)

Using the latest cached version of the module from /home/studio-lab-user/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--rouge/b01e0accf3bd6dd24839b769a5fda24e14995071570870922c71970b3a6ed886 (last modified on Wed Jul 30 18:48:05 2025) since it couldn't be found locally at evaluate-metric--rouge, or remotely on the Hugging Face Hub.


ModuleNotFoundError: No module named 'rouge_score'

### Visualization

In [ ]:
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

# ROUGE Scores analysis
plt.figure(figsize=(12, 8))
barWidth = 0.25
r1 = np.arange(len(metrics))
r2 = [x + barWidth for x in r1]
r3 = [x + barWidth for x in r2]

plt.bar(r1, rouge_results_df['Zero-shot'], width=barWidth, label='Zero-shot')
plt.bar(r2, rouge_results_df['One-shot'], width=barWidth, label='One-shot')
plt.bar(r3, rouge_results_df['Few-shot'], width=barWidth, label='Few-shot')


plt.xlabel('Metrics')
plt.ylabel('Score')
plt.title('ROUGE Scores by Summarization Approach')
plt.xticks([r + barWidth for r in range(len(metrics))], metrics)
plt.legend()

plt.savefig(f"{results_dir}/rouge_scores.png")
plt.show()

# Summary length analysis
summary_lengths_df = pd.DataFrame({
    "Example": range(1, len(test_examples) + 1),
    "Reference": [len(row["Summary"].split()) for _, row in df.iterrows()],
    "Zero-shot": [len(row["Zero-shot Summary"].split()) for _, row in df.iterrows()],
    "One-shot": [len(row["One-shot Summary"].split()) for _, row in df.iterrows()],
    "Few-shot": [len(row["Few-shot Summary"].split()) for _, row in df.iterrows()],
})

# Melt the DataFrame for easier plotting
melted = pd.melt(summary_lengths_df, id_vars=['Example'], 
                 value_vars=['Reference', 'Zero-shot', 'One-shot', 'Few-shot'],
                 var_name='Method', value_name='Word Count')

plt.figure(figsize=(12, 8))
bar_plot = plt.bar(melted['Method'] + ' ' + melted['Example'].astype(str), melted['Word Count'], 
       color=['grey', 'blue', 'green', 'red'] * len(test_examples))
plt.title('Summary Length by Method and Example')
plt.xlabel('Method and Example Number')
plt.ylabel('Word Count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f"{results_dir}/summary_lengths.png")
plt.show()

### Save Results

In [ ]:
plt.close("all")

results_df.to_csv(f'{results_dir}/dialogue_summarization_results.csv', index=False)
rouge_results_df.to_csv(f'{results_dir}/rouge_scores.csv', index=False)
summary_lengths_df.to_csv(f'{results_dir}/summary_lengths.csv', index=False)